<a href="https://colab.research.google.com/github/boss-defender/Born-Baby-Ai/blob/main/Just_born_Baby_Ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 👶 Universal Baby AI: Frontier & Multi-Scale DeepSeek V4.1 Flash Architecture

An ultra-flexible, modular, and scalable Deep Learning framework engineered to scale from **100k-parameter Nano models** (for micro-tasks and embedded devices) to **Frontier-scale Foundation models** (for Coding, Mathematics, Science, and General Chat):
- **Elastic Model Scales**: From `Baby-Nano` (~350K params) and `Baby-Tiny` (~15M params) to `Baby-Flash` (~65M params) and `Baby-Pro` (~180M params) or full `Custom` dimensions.
- **Universal Domain Ingestion**: Automatic detection and formatting for **General Chat**, **Coding**, **Mathematics/Reasoning** (`<think>`), **Science/STEM**, and **Raw Pretraining**.
- **DeepSeek V4.1 Flash Backbone**: Dynamic Sparse MoE (Top-2 Routed + Shared Expert), Multi-Head Latent Attention (MLA), Decoupled RoPE, and Unified Causal Encoder-Decoder.
- **Frontier-Grade Training**: Gradient Accumulation for scalable batching, Cosine Annealing with Warmup, and Google Drive auto-resume with rolling 3-checkpoint storage isolation.
- **Production Release**: Complete Hugging Face repository package (`model.safetensors`, `config.json`, tokenizer) saved locally in `./baby_ai_model/` + 1-Click GGUF & Ollama export for LM Studio and Ollama.

# **⚠️ Caution:**
**📢 1. With colab free tier , you can train with smaller datasets or lower max samples.**

**‼️ 2. For different datasets , you may need to make minor edits to the code (specially cell 2).**

**🚩 3. Make sure to give correct max-samples integer or leave it empty.**


In [3]:
# @title ⚙️ Cell 1: Dependencies, Automated Drive Mounting & Universal UI Form { run: "auto" }
# @markdown Configure your model scale, task domain, dataset, and training parameters using the visual form below.

# @markdown ### 🧠 Model Identifier & Elastic Capacity
MODEL_SCALE = "Baby-Flash"  # @param ["Baby-Nano", "Baby-Tiny", "Baby-Flash", "Baby-Pro", "Custom"]
CUSTOM_HIDDEN_SIZE = 256  # @param {type:"integer"}
CUSTOM_NUM_LAYERS = 6  # @param {type:"integer"}
CUSTOM_NUM_EXPERTS = 8  # @param {type:"integer"}

# @markdown ### 🎯 Task Domain & Specialization
TASK_DOMAIN = "auto-detect"  # @param ["auto-detect", "general_chat", "coding", "mathematics_reasoning", "science_stem", "vision", "raw_pretrain"]

# @markdown ### 📂 Dataset Source & Location
DATASET_SOURCE = "Hugging Face"  # @param ["Hugging Face", "Custom Upload / Drive"]
DATASET_PATH = "Akhil391/daily_dialog"  # @param {type:"string"}
MAX_SAMPLES = None  # @param {type:"integer"}
MAX_SEQ_LENGTH = 128  # @param {type:"integer"}

# @markdown ### 💾 Google Drive Smart Checkpointing (Training State Only)
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
SAVE_EVERY_N_STEPS = 200  # @param {type:"integer"}
MAX_CHECKPOINTS_TO_KEEP = 3  # @param {type:"integer"}

# @markdown ### 📦 Local Export Directory (Hugging Face / GGUF Standard Package)
LOCAL_EXPORT_DIR = "./baby_ai_model"  # @param {type:"string"}

# @markdown ### 🚀 Scalable Training Controls
BATCH_SIZE = 8  # @param [2, 4, 8, 16, 32]
GRAD_ACCUM_STEPS = 1  # @param [1, 2, 4, 8, 16] - Simulate large batches without OOM
EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 3e-4  # @param {type:"number"}
ENABLE_MIXED_PRECISION = True  # @param {type:"boolean"}

# @markdown ### 🌐 Hugging Face Hub (Optional Publishing)
PUSH_TO_HUB = False  # @param {type:"boolean"}
HF_TOKEN = ""  # @param {type:"string"}
HF_REPO_ID = "my-babyflash-model"  # @param {type:"string"}

# 1. Silent Automated Dependency Installation
print("Installing core dependencies (silently)...")
import subprocess
import sys
import os

deps = ["transformers", "datasets", "accelerate", "sentencepiece", "pillow", "einops", "safetensors", "gguf"]
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + deps)
    print("✓ Dependencies verified (transformers, datasets, accelerate, safetensors, gguf).")
except Exception as e:
    print(f"Notice: Dependency installation message: {e}")

# 2. Automated Google Drive Mount Engine (Training Resume State Only)
BASE_CHECKPOINT_DIR = "./BabyAI_Checkpoints"
if SAVE_TO_DRIVE:
    try:
        if 'google.colab' in sys.modules or os.path.exists('/content'):
            from google.colab import drive
            drive.mount('/content/drive')
            BASE_CHECKPOINT_DIR = "/content/drive/MyDrive/BabyAI_Checkpoints"
            print("✓ Google Drive mounted at /content/drive")
            print(f"✓ Training Checkpoints (Resume State) will sync to: {BASE_CHECKPOINT_DIR}")
        else:
            print("Notice: Local environment detected. Checkpoints will save to ./BabyAI_Checkpoints")
    except Exception as e:
        print(f"Notice: Google Drive mount skipped ({e}). Checkpoints save to ./BabyAI_Checkpoints")

os.makedirs(BASE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOCAL_EXPORT_DIR, exist_ok=True)

# 3. Hardware Audit
import torch
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Active Compute Device: {device.upper()}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Hardware: {torch.cuda.get_device_name(0)} ({vram_gb:.2f} GB VRAM)")
    torch.cuda.empty_cache()
else:
    print("Mode: CPU (Low-memory guardrails and gradient checkpoints active)")
print(f"Configured Scale: {MODEL_SCALE} | Effective Batch Size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print("=" * 60)


Installing core dependencies (silently)...
✓ Dependencies verified (transformers, datasets, accelerate, safetensors, gguf).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted at /content/drive
✓ Training Checkpoints (Resume State) will sync to: /content/drive/MyDrive/BabyAI_Checkpoints
PyTorch Version: 2.11.0+cu128
Active Compute Device: CUDA
GPU Hardware: Tesla T4 (14.56 GB VRAM)
Configured Scale: Baby-Flash | Effective Batch Size: 8


In [2]:
# @title 📊 Cell 2: Universal Data Ingestion & Domain Auto-Classifier
Run_This_Cell= "2" # @param {type:"string"}
import os
import re
from typing import Dict, Any, List, Tuple
from datasets import load_dataset
from transformers import AutoTokenizer

# 1. Directory Sanitization & Unique Run Identification
def sanitize_identifier(name: str) -> str:
    cleaned = re.sub(r'[^a-zA-Z0-9_]', '_', str(name).strip())
    cleaned = re.sub(r'_+', '_', cleaned).strip('_')
    return cleaned or "unnamed"

UNIQUE_RUN_ID = f"{sanitize_identifier(MODEL_SCALE)}_on_{sanitize_identifier(DATASET_PATH)}"
RUN_CHECKPOINT_DIR = os.path.join(BASE_CHECKPOINT_DIR, UNIQUE_RUN_ID)
os.makedirs(RUN_CHECKPOINT_DIR, exist_ok=True)

print("=" * 60)
print(f"✓ Unique Training Run ID: '{UNIQUE_RUN_ID}'")
print(f"✓ Drive Checkpoint Folder (Resume Only): '{RUN_CHECKPOINT_DIR}'")
print(f"✓ Local Release Package Folder: '{LOCAL_EXPORT_DIR}'")
print("=" * 60)

# 2. Tokenizer Setup with Standard ChatML, Reasoning & Special Tokens
TOKENIZER_NAME = "Qwen/Qwen2.5-0.5B"
print(f"Loading tokenizer: {TOKENIZER_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

SPECIAL_TOKENS = {
    "additional_special_tokens": [
        "<|im_start|>", "<|im_end|>",              # ChatML delimiters
        "<think>", "</think>",                      # DeepSeek Reasoning tokens
        "<tool_call>", "</tool_call>",              # Agent tool invocation tokens
        "<tool_response>", "</tool_response>",      # Agent tool response tokens
        "<image>", "</image>",                      # Vision patch anchor tokens
    ]
}
num_added = tokenizer.add_special_tokens(SPECIAL_TOKENS)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

TOTAL_VOCAB_SIZE = max(len(tokenizer), tokenizer.vocab_size)
print(f"✓ Tokenizer ready with {num_added} special tokens | Total Vocab Size: {TOTAL_VOCAB_SIZE:,}")

# 3. Universal Domain Classifier & Adaptive Prompt Formatter
def detect_domain(cols: set, row: Dict[str, Any]) -> str:
    if TASK_DOMAIN != "auto-detect":
        return TASK_DOMAIN

    col_names = " ".join(cols).lower()
    if any(k in col_names for k in ["code", "python", "solution_code", "func", "programming"]):
        return "coding"
    if any(k in col_names for k in ["problem", "gsm8k", "math", "proof", "equation", "steps"]):
        return "mathematics_reasoning"
    if any(k in col_names for k in ["dialog", "dialogue", "conversations", "messages", "chat", "turns"]):
        return "general_chat"
    if any(k in col_names for k in ["instruction", "prompt", "question"]) and any(k in col_names for k in ["output", "response", "answer", "completion"]):
        return "general_chat"
    if any(k in col_names for k in ["image", "pixel_values", "caption"]):
        return "vision"
    return "raw_pretrain"

def format_sample(row: Dict[str, Any]) -> str:
    cols = set(row.keys())
    domain = detect_domain(cols, row)

    # A. Multi-turn Dialogue Dataset (Supports messages list-of-dicts, ShareGPT, dialog list-of-strings)
    diag_col = next((c for c in ["messages", "conversations", "dialog", "dialogue", "chat", "turns"] if c in cols), None)
    if diag_col and isinstance(row[diag_col], (list, tuple)):
        turns = row[diag_col]
        if turns:
            formatted_convo = []
            for i, turn in enumerate(turns):
                if isinstance(turn, dict):
                    raw_role = str(turn.get("role") or turn.get("from") or ("user" if i % 2 == 0 else "assistant")).lower()
                    if raw_role in ["human", "user", "input"]:
                        role = "user"
                    elif raw_role in ["gpt", "assistant", "bot", "output", "model"]:
                        role = "assistant"
                    elif raw_role in ["system"]:
                        role = "system"
                    else:
                        role = raw_role
                    content = str(turn.get("content") or turn.get("value") or turn.get("text") or "").strip()
                else:
                    role = "user" if (i % 2 == 0) else "assistant"
                    content = str(turn).strip()

                if content:
                    formatted_convo.append(f"<|im_start|>{role}\n{content}<|im_end|>")
            if formatted_convo:
                return "\n".join(formatted_convo)

    # Find standard Q&A columns
    inst = next((c for c in ["instruction", "prompt", "question", "problem", "query", "input_text"] if c in cols), None)
    out = next((c for c in ["output", "response", "answer", "solution", "completion", "code", "target"] if c in cols), None)
    inp = next((c for c in ["input", "context", "rationale"] if c in cols), None)

    q_text = str(row[inst]).strip() if inst and row[inst] else ""
    a_text = str(row[out]).strip() if out and row[out] else ""
    c_text = f"\nContext:\n{row[inp]}" if inp and row[inp] else ""

    # B. Coding Domain
    if domain == "coding":
        req = q_text or "Write code for this task."
        return f"<|im_start|>user\n{req}{c_text}<|im_end|>\n<|im_start|>assistant\n```python\n{a_text}\n```<|im_end|>"

    # C. Mathematics & Logical Reasoning Domain (DeepSeek <think> reasoning)
    elif domain == "mathematics_reasoning":
        req = q_text or "Solve this mathematical reasoning problem."
        return f"<|im_start|>user\n{req}{c_text}<|im_end|>\n<|im_start|>assistant\n<think>\nAnalyzing mathematical constraints and calculating step-by-step.\n</think>\n{a_text}<|im_end|>"

    # D. Science / STEM Domain
    elif domain == "science_stem":
        req = q_text or "Explain the scientific principles involved."
        return f"<|im_start|>user\n{req}{c_text}<|im_end|>\n<|im_start|>assistant\n{a_text}<|im_end|>"

    # E. General Chat & Instruction
    elif domain == "general_chat":
        if q_text and a_text:
            return f"<|im_start|>user\n{q_text}{c_text}<|im_end|>\n<|im_start|>assistant\n{a_text}<|im_end|>"

    # F. Vision Multi-modal
    elif domain == "vision":
        cap = str(row.get("text") or row.get("caption") or row.get("response") or "A descriptive visual scene.").strip()
        return f"<image> Describe image: {cap}<|im_end|>"

    # G. Raw Continuous Pretraining (Books, Wiki, Stories)
    text_col = next((c for c in ["text", "content", "body", "article", "story"] if c in cols), None)
    if text_col and row[text_col]:
        return f"{str(row[text_col]).strip()}<|im_end|>"

    # H. Universal Fallback
    vals = [str(v).strip() for v in row.values() if v is not None and not isinstance(v, (dict, list))]
    return " ".join(vals) + "<|im_end|>"

# 4. Universal Fault-Tolerant Dataset Ingestion (Auto-Detects Any Split, Subsets & Remote Code)
print(f"\nIngesting dataset via Mode: '{DATASET_SOURCE}' from '{DATASET_PATH}'...")

def load_any_dataset(path_or_name: str, source_type: str):
    # A. Local files / Google Drive files
    if source_type == "Custom Upload / Drive" or os.path.exists(path_or_name):
        ext = os.path.splitext(path_or_name)[-1].lower()
        type_map = {".json": "json", ".jsonl": "json", ".csv": "csv", ".tsv": "csv", ".parquet": "parquet", ".txt": "text"}
        file_type = type_map.get(ext, "text")
        try:
            return load_dataset(file_type, data_files=path_or_name, split="train")
        except Exception:
            ds_dict = load_dataset(file_type, data_files=path_or_name)
            return ds_dict[list(ds_dict.keys())[0]]

    # B. Hugging Face Hub (Handles trust_remote_code for Akhil391/daily_dialog, subsets, and split auto-selection)
    for trc in [True, False]:
        try:
            return load_dataset(path_or_name, split="train", trust_remote_code=trc)
        except Exception as e:
            err = str(e).lower()
            if "config" in err or "builder" in err:
                try:
                    from datasets import get_dataset_config_names
                    cfgs = get_dataset_config_names(path_or_name, trust_remote_code=trc)
                    if cfgs:
                        print(f"✓ Auto-selected dataset subset: '{cfgs[0]}'")
                        return load_dataset(path_or_name, cfgs[0], split="train", trust_remote_code=trc)
                except Exception:
                    pass

    # Inspect all available splits if 'train' is missing (e.g. 'train_sft', 'data', 'validation')
    for trc in [True, False]:
        try:
            ds_dict = load_dataset(path_or_name, trust_remote_code=trc)
            if hasattr(ds_dict, "keys"):
                available_splits = list(ds_dict.keys())
                chosen_split = next((s for s in available_splits if "train" in s.lower()), None)
                if not chosen_split:
                    chosen_split = next((s for s in available_splits if any(k in s.lower() for k in ["sft", "data", "prompt", "chat"])), None)
                if not chosen_split:
                    chosen_split = available_splits[0]
                print(f"✓ Auto-detected split '{chosen_split}' from available: {available_splits}")
                return ds_dict[chosen_split]
            return ds_dict
        except Exception:
            pass

    return load_dataset(path_or_name, split="train", trust_remote_code=True)

raw_dataset = load_any_dataset(DATASET_PATH, DATASET_SOURCE)

if MAX_SAMPLES is not None and len(raw_dataset) > MAX_SAMPLES:
    raw_dataset = raw_dataset.shuffle(seed=42).select(range(MAX_SAMPLES))

detected_domain = detect_domain(set(raw_dataset[0].keys()), raw_dataset[0])
print(f"✓ Ingested {len(raw_dataset):,} samples | Active Domain: '{detected_domain.upper()}'")
print("\n--- Sample Formatted Input Preview ---")
preview_str = format_sample(raw_dataset[0])
print(preview_str[:300] + ("..." if len(preview_str) > 300 else "") + "\n" + "-" * 38)

def tokenize_batch(examples):
    keys = list(examples.keys())
    batch_len = len(examples[keys[0]])
    formatted = []
    for i in range(batch_len):
        row = {k: examples[k][i] for k in keys}
        formatted.append(format_sample(row))

    tokens = tokenizer(
        formatted,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        return_tensors=None,
    )
    return {
        "input_ids": tokens["input_ids"],
        "attention_mask": tokens["attention_mask"],
    }

tokenized_dataset = raw_dataset.map(
    tokenize_batch,
    batched=True,
    batch_size=1000,
    remove_columns=raw_dataset.column_names,
    desc="Tokenizing for BabyFlash Causal LM",
)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])
print(f"✓ Pipeline tokenized {len(tokenized_dataset):,} samples. Tensor shape: {tokenized_dataset[0]['input_ids'].shape}")


✓ Unique Training Run ID: 'Baby_Flash_on_Akhil391_daily_dialog'
✓ Drive Checkpoint Folder (Resume Only): '/content/drive/MyDrive/BabyAI_Checkpoints/Baby_Flash_on_Akhil391_daily_dialog'
✓ Local Release Package Folder: './baby_ai_model'
Loading tokenizer: Qwen/Qwen2.5-0.5B...


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

✓ Tokenizer ready with 8 special tokens | Total Vocab Size: 151,671

Ingesting dataset via Mode: 'Hugging Face' from 'Akhil391/daily_dialog'...


README.md:   0%|          | 0.00/7.27k [00:00<?, ?B/s]

daily_dialog.py:   0%|          | 0.00/4.85k [00:00<?, ?B/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Akhil391/daily_dialog' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Akhil391/daily_dialog' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Akhil391/daily_dialog' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_c

RuntimeError: Dataset scripts are no longer supported, but found daily_dialog.py

In [ ]:
# @title 🧠 Cell 3: Fully-Implemented PyTorch DeepSeek Engine (Multi-Scale)
Run_This_Cell= "3" # @param {type:"string"}
import math
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict, Union
from transformers.configuration_utils import PretrainedConfig
from transformers.modeling_utils import PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast

# 1. Elastic Architecture Scaling Configurations (100k params to Frontier Scale)
SCALE_CONFIGS = {
    "Baby-Nano": {
        "hidden_size": 96,
        "num_encoder_layers": 1,
        "num_decoder_layers": 2,
        "num_attention_heads": 4,
        "kv_compression_dim": 24,
        "rope_dim": 12,
        "intermediate_size": 192,
        "encoder_latent_dim": 48,
        "num_routed_experts": 4,
        "num_experts_per_tok": 1,
        "num_shared_experts": 1,
    },
    "Baby-Tiny": {
        "hidden_size": 192,
        "num_encoder_layers": 2,
        "num_decoder_layers": 4,
        "num_attention_heads": 6,
        "kv_compression_dim": 48,
        "rope_dim": 16,
        "intermediate_size": 384,
        "encoder_latent_dim": 96,
        "num_routed_experts": 8,
        "num_experts_per_tok": 2,
        "num_shared_experts": 1,
    },
    "Baby-Flash": {
        "hidden_size": 256,
        "num_encoder_layers": 4,
        "num_decoder_layers": 6,
        "num_attention_heads": 8,
        "kv_compression_dim": 64,
        "rope_dim": 16,
        "intermediate_size": 512,
        "encoder_latent_dim": 128,
        "num_routed_experts": 8,
        "num_experts_per_tok": 2,
        "num_shared_experts": 1,
    },
    "Baby-Pro": {
        "hidden_size": 512,
        "num_encoder_layers": 6,
        "num_decoder_layers": 10,
        "num_attention_heads": 8,
        "kv_compression_dim": 128,
        "rope_dim": 32,
        "intermediate_size": 1024,
        "encoder_latent_dim": 256,
        "num_routed_experts": 16,
        "num_experts_per_tok": 2,
        "num_shared_experts": 2,
    },
    "Custom": {
        "hidden_size": CUSTOM_HIDDEN_SIZE,
        "num_encoder_layers": max(1, CUSTOM_NUM_LAYERS // 2),
        "num_decoder_layers": CUSTOM_NUM_LAYERS,
        "num_attention_heads": 8,
        "kv_compression_dim": CUSTOM_HIDDEN_SIZE // 4,
        "rope_dim": 16,
        "intermediate_size": CUSTOM_HIDDEN_SIZE * 2,
        "encoder_latent_dim": CUSTOM_HIDDEN_SIZE // 2,
        "num_routed_experts": CUSTOM_NUM_EXPERTS,
        "num_experts_per_tok": 2,
        "num_shared_experts": 1,
    }
}
ACTIVE_SCALE = SCALE_CONFIGS.get(MODEL_SCALE, SCALE_CONFIGS["Baby-Flash"])

class BabyFlashConfig(PretrainedConfig):
    model_type = "baby_flash"
    def __init__(
        self,
        vocab_size: int = 151665,
        hidden_size: int = ACTIVE_SCALE["hidden_size"],
        num_encoder_layers: int = ACTIVE_SCALE["num_encoder_layers"],
        num_decoder_layers: int = ACTIVE_SCALE["num_decoder_layers"],
        num_attention_heads: int = ACTIVE_SCALE["num_attention_heads"],
        kv_compression_dim: int = ACTIVE_SCALE["kv_compression_dim"],
        rope_dim: int = ACTIVE_SCALE["rope_dim"],
        intermediate_size: int = ACTIVE_SCALE["intermediate_size"],
        encoder_latent_dim: int = ACTIVE_SCALE["encoder_latent_dim"],
        num_routed_experts: int = ACTIVE_SCALE["num_routed_experts"],
        num_experts_per_tok: int = ACTIVE_SCALE["num_experts_per_tok"],
        num_shared_experts: int = ACTIVE_SCALE["num_shared_experts"],
        router_aux_loss_coef: float = 0.01,
        image_size: int = 224,
        patch_size: int = 16,
        max_position_embeddings: int = 2048,
        initializer_range: float = 0.02,
        rms_norm_eps: float = 1e-6,
        use_cache: bool = True,
        pad_token_id: int = 151643,
        bos_token_id: int = 151643,
        eos_token_id: int = 151643,
        tie_word_embeddings: bool = True,
        **kwargs,
    ):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_encoder_layers = num_encoder_layers
        self.num_decoder_layers = num_decoder_layers
        self.num_attention_heads = num_attention_heads
        self.kv_compression_dim = kv_compression_dim
        self.rope_dim = rope_dim
        self.intermediate_size = intermediate_size
        self.encoder_latent_dim = encoder_latent_dim
        self.num_routed_experts = num_routed_experts
        self.num_experts_per_tok = num_experts_per_tok
        self.num_shared_experts = num_shared_experts
        self.router_aux_loss_coef = router_aux_loss_coef
        self.image_size = image_size
        self.patch_size = patch_size
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.rms_norm_eps = rms_norm_eps
        self.use_cache = use_cache
        super().__init__(
            pad_token_id=pad_token_id,
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            tie_word_embeddings=tie_word_embeddings,
            **kwargs,
        )
        self.auto_map = {
            "AutoConfig": "configuration_babyflash.BabyFlashConfig",
            "AutoModelForCausalLM": "modeling_babyflash.BabyFlashMultiModalForCausalLM"
        }
        self.architectures = ["BabyFlashMultiModalForCausalLM"]

# 2. Normalization & Positional Embeddings
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_position_embeddings: int = 2048, base: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len_cached = max_position_embeddings
        t = torch.arange(self.max_seq_len_cached, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, seq_len: int, device: torch.device, offset: int = 0) -> Tuple[torch.Tensor, torch.Tensor]:
        total_len = seq_len + offset
        if total_len > self.max_seq_len_cached:
            t = torch.arange(total_len, device=device, dtype=torch.float32)
            freqs = torch.outer(t, self.inv_freq.to(device))
            emb = torch.cat((freqs, freqs), dim=-1)
            cos = emb.cos()[None, None, :, :]
            sin = emb.sin()[None, None, :, :]
            return cos[:, :, offset:total_len, :], sin[:, :, offset:total_len, :]
        return (
            self.cos_cached[:, :, offset:total_len, :].to(device),
            self.sin_cached[:, :, offset:total_len, :].to(device),
        )

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_emb(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    return (x * cos) + (rotate_half(x) * sin)

# 3. Vision Projection Layer
class VisionPatchEmbedding(nn.Module):
    def __init__(self, config: BabyFlashConfig):
        super().__init__()
        self.patch_size = config.patch_size
        self.proj = nn.Conv2d(3, config.hidden_size, kernel_size=self.patch_size, stride=self.patch_size)
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        patches = self.proj(pixel_values).flatten(2).transpose(1, 2)
        return self.norm(patches)

# 4. Multi-Head Latent Attention (MLA) with KV Compression
class CompressedAttention(nn.Module):
    def __init__(self, config: BabyFlashConfig, layer_idx: int = 0):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.kv_comp_dim = config.kv_compression_dim
        self.rope_dim = config.rope_dim

        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.q_rope_proj = nn.Linear(self.hidden_size, self.num_heads * self.rope_dim, bias=False)
        self.kv_down_proj = nn.Linear(self.hidden_size, self.kv_comp_dim, bias=False)
        self.kv_norm = RMSNorm(self.kv_comp_dim, eps=config.rms_norm_eps)
        self.k_up_proj = nn.Linear(self.kv_comp_dim, self.num_heads * self.head_dim, bias=False)
        self.v_up_proj = nn.Linear(self.kv_comp_dim, self.num_heads * self.head_dim, bias=False)
        self.k_rope_proj = nn.Linear(self.hidden_size, self.rope_dim, bias=False)
        self.out_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)
        self.rotary = RotaryEmbedding(self.rope_dim, max_position_embeddings=config.max_position_embeddings)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        past_key_value: Optional[Dict[str, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Dict[str, torch.Tensor]]]:
        B, T, _ = hidden_states.shape
        q_c = self.q_proj(hidden_states).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        q_r = self.q_rope_proj(hidden_states).view(B, T, self.num_heads, self.rope_dim).transpose(1, 2)

        c_kv = self.kv_norm(self.kv_down_proj(hidden_states))
        k_r = self.k_rope_proj(hidden_states).view(B, T, 1, self.rope_dim).transpose(1, 2)

        offset = 0
        if past_key_value is not None and "c_kv" in past_key_value:
            offset = past_key_value["c_kv"].shape[1]
            c_kv = torch.cat([past_key_value["c_kv"], c_kv], dim=1)
            k_r = torch.cat([past_key_value["k_rope"], k_r], dim=2)

        present_key_value = {"c_kv": c_kv, "k_rope": k_r} if use_cache else None
        total_len = c_kv.shape[1]

        cos, sin = self.rotary(total_len, device=hidden_states.device)
        q_r = apply_rotary_emb(q_r, cos[:, :, offset : offset + T, :], sin[:, :, offset : offset + T, :])
        k_r_rotated = apply_rotary_emb(k_r, cos, sin)

        k_c = self.k_up_proj(c_kv).view(B, total_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_up_proj(c_kv).view(B, total_len, self.num_heads, self.head_dim).transpose(1, 2)

        k_r_expanded = k_r_rotated.repeat(1, self.num_heads, 1, 1)
        q = torch.cat([q_c, q_r], dim=-1)
        k = torch.cat([k_c, k_r_expanded], dim=-1)

        is_causal = (T > 1 and past_key_value is None and attention_mask is None)
        attn_mask = None
        if attention_mask is not None and attention_mask.dim() == 2:
            attn_mask = attention_mask[:, None, None, :].to(dtype=q.dtype)
            attn_mask = (1.0 - attn_mask) * -10000.0

        attn_out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, is_causal=is_causal)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, self.num_heads * self.head_dim)
        return self.out_proj(attn_out), present_key_value

# 5. SwiGLU & Sparse MoE Layer
class SwiGLU(nn.Module):
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        self.gate = nn.Linear(dim, hidden_dim, bias=False)
        self.up = nn.Linear(dim, hidden_dim, bias=False)
        self.down = nn.Linear(dim, hidden_dim, bias=False)
        self.out = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.out(F.silu(self.gate(x)) * self.up(x))

class MoELayer(nn.Module):
    def __init__(self, config: BabyFlashConfig):
        super().__init__()
        self.config = config
        self.num_routed = config.num_routed_experts
        self.top_k = min(config.num_experts_per_tok, self.num_routed)

        self.router = nn.Linear(config.hidden_size, self.num_routed, bias=False)
        self.routed_experts = nn.ModuleList([
            SwiGLU(config.hidden_size, config.intermediate_size)
            for _ in range(self.num_routed)
        ])
        self.shared_experts = nn.ModuleList([
            SwiGLU(config.hidden_size, config.intermediate_size)
            for _ in range(config.num_shared_experts)
        ])
        self.aux_coef = config.router_aux_loss_coef

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, C = x.shape
        x_flat = x.view(-1, C)
        router_logits = self.router(x_flat)
        router_probs = F.softmax(router_logits, dim=-1)

        weights, indices = torch.topk(router_probs, self.top_k, dim=-1)
        weights = weights / (weights.sum(dim=-1, keepdim=True) + 1e-6)

        if self.training and self.aux_coef > 0:
            mask = torch.zeros_like(router_probs).scatter_(1, indices, 1.0)
            f_avg = mask.mean(dim=0)
            P_avg = router_probs.mean(dim=0)
            aux_loss = self.aux_coef * self.num_routed * (P_avg * f_avg).sum()
        else:
            aux_loss = torch.tensor(0.0, device=x.device, dtype=x.dtype)

        routed_out = torch.zeros_like(x_flat)
        for exp_id, expert in enumerate(self.routed_experts):
            for k_idx in range(self.top_k):
                sel = (indices[:, k_idx] == exp_id)
                if sel.any():
                    t_idx = torch.nonzero(sel).squeeze(-1)
                    w = weights[sel, k_idx].unsqueeze(-1)
                    exp_out = expert(x_flat[sel])
                    routed_out.index_put_((t_idx,), exp_out * w, accumulate=True)

        shared_out = torch.zeros_like(x_flat)
        for shared in self.shared_experts:
            shared_out = shared_out + shared(x_flat)

        return (routed_out + shared_out).view(B, T, C), aux_loss

# 6. Causal Encoder & Decoder Blocks
class CausalEncoderBlock(nn.Module):
    def __init__(self, config: BabyFlashConfig, layer_idx: int = 0):
        super().__init__()
        self.norm1 = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.attn = CompressedAttention(config, layer_idx=layer_idx)
        self.norm2 = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.ffn = SwiGLU(config.hidden_size, config.intermediate_size)

    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        attn_out, _ = self.attn(self.norm1(x), attention_mask=attention_mask, use_cache=False)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x

class CausalDecoderBlock(nn.Module):
    def __init__(self, config: BabyFlashConfig, layer_idx: int = 0):
        super().__init__()
        self.norm1 = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.attn = CompressedAttention(config, layer_idx=layer_idx)
        self.norm2 = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.moe = MoELayer(config)

    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None, past_key_value: Optional[Dict[str, torch.Tensor]] = None, use_cache: bool = False):
        attn_out, present_kv = self.attn(self.norm1(x), attention_mask=attention_mask, past_key_value=past_key_value, use_cache=use_cache)
        x = x + attn_out
        moe_out, aux_loss = self.moe(self.norm2(x))
        x = x + moe_out
        return x, aux_loss, present_kv

# 7. Complete BabyFlash Model
@dataclass
class BabyFlashOutput(CausalLMOutputWithPast):
    loss: Optional[torch.FloatTensor] = None
    logits: torch.FloatTensor = None
    past_key_values: Optional[List[Dict[str, torch.Tensor]]] = None
    router_aux_loss: Optional[torch.FloatTensor] = None

class BabyFlashMultiModalModel(nn.Module):
    def __init__(self, config: BabyFlashConfig):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.embed_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.vision_proj = VisionPatchEmbedding(config)

        self.encoder = nn.ModuleList([CausalEncoderBlock(config, i) for i in range(config.num_encoder_layers)])
        self.enc_to_dec = nn.Linear(config.hidden_size, config.encoder_latent_dim, bias=False)
        self.dec_latent = nn.Linear(config.encoder_latent_dim, config.hidden_size, bias=False)
        self.decoder = nn.ModuleList([CausalDecoderBlock(config, i) for i in range(config.num_decoder_layers)])
        self.final_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        pixel_values: Optional[torch.FloatTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        past_key_values: Optional[List[Dict[str, torch.Tensor]]] = None,
        use_cache: bool = True
    ):
        h = self.embed_norm(self.embed_tokens(input_ids)) if input_ids is not None else None
        if pixel_values is not None:
            vis = self.vision_proj(pixel_values)
            if h is not None:
                h = torch.cat([vis, h], dim=1)
                if attention_mask is not None:
                    vm = torch.ones((pixel_values.shape[0], vis.shape[1]), device=pixel_values.device)
                    attention_mask = torch.cat([vm, attention_mask], dim=1)
            else:
                h = vis

        total_aux = torch.tensor(0.0, device=h.device, dtype=h.dtype)
        present_kvs = [] if use_cache else None

        # Causal Encoder: smooth representations across both prefill and generation
        enc_h = h
        for enc_layer in self.encoder:
            enc_h = enc_layer(enc_h, attention_mask=attention_mask)
        h = h + self.dec_latent(self.enc_to_dec(enc_h))

        # Causal Decoder Stack with MoE
        for idx, dec_layer in enumerate(self.decoder):
            p_kv = past_key_values[idx] if past_key_values is not None else None
            h, aux_loss, cur_kv = dec_layer(h, attention_mask=attention_mask, past_key_value=p_kv, use_cache=use_cache)
            total_aux = total_aux + aux_loss
            if use_cache:
                present_kvs.append(cur_kv)

        return self.final_norm(h), total_aux, present_kvs

class BabyFlashMultiModalForCausalLM(PreTrainedModel):
    config_class = BabyFlashConfig
    base_model_prefix = "model"
    _tied_weights_keys = ["lm_head.weight"]

    def __init__(self, config: BabyFlashConfig):
        super().__init__(config)
        self.model = BabyFlashMultiModalModel(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        if config.tie_word_embeddings:
            self.lm_head.weight = self.model.embed_tokens.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)

    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        pixel_values: Optional[torch.FloatTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        past_key_values: Optional[List[Dict[str, torch.Tensor]]] = None,
        labels: Optional[torch.LongTensor] = None,
        use_cache: bool = True,
        return_dict: bool = True,
        **kwargs,
    ):
        h, aux_loss, present_kvs = self.model(input_ids, pixel_values, attention_mask, past_key_values, use_cache)
        logits = self.lm_head(h)

        loss = None
        if labels is not None:
            s_logits = logits[..., :-1, :].contiguous()
            s_labels = labels[..., 1:].contiguous()
            if s_labels.shape[1] < s_logits.shape[1]:
                s_logits = s_logits[:, s_logits.shape[1] - s_labels.shape[1]:, :]
            lm_loss = F.cross_entropy(s_logits.view(-1, self.config.vocab_size), s_labels.view(-1), ignore_index=-100)
            loss = lm_loss + aux_loss

        return BabyFlashOutput(loss=loss, logits=logits, past_key_values=present_kvs, router_aux_loss=aux_loss)

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.LongTensor,
        pixel_values: Optional[torch.FloatTensor] = None,
        max_new_tokens: int = 60,
        temperature: float = 0.7,
        top_k: int = 50,
        top_p: float = 0.9,
        repetition_penalty: float = 1.25,
        eos_token_id: Optional[int] = None,
    ) -> torch.LongTensor:
        self.eval()
        eos_ids = set()
        if eos_token_id is not None:
            eos_ids.add(eos_token_id)
        if self.config.eos_token_id is not None:
            eos_ids.add(self.config.eos_token_id)
        im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
        if im_end_id is not None and im_end_id != tokenizer.unk_token_id:
            eos_ids.add(im_end_id)

        gen = input_ids.clone()
        p_kvs = None
        curr_ids = input_ids
        first = True

        for _ in range(max_new_tokens):
            if first and pixel_values is not None:
                out = self(input_ids=curr_ids, pixel_values=pixel_values, past_key_values=p_kvs, use_cache=True)
                first = False
            else:
                out = self(input_ids=curr_ids, past_key_values=p_kvs, use_cache=True)

            logits = out.logits[:, -1, :].clone()
            p_kvs = out.past_key_values

            # Anti-Repetition Penalty Masking
            if repetition_penalty != 1.0:
                for b in range(logits.shape[0]):
                    prev_tokens = set(gen[b].tolist())
                    for t_id in prev_tokens:
                        if logits[b, t_id] > 0:
                            logits[b, t_id] /= repetition_penalty
                        else:
                            logits[b, t_id] *= repetition_penalty

            if temperature > 0:
                logits = logits / max(temperature, 1e-4)
                if top_k > 0:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float("Inf")
                if 0.0 < top_p < 1.0:
                    s_logits, s_indices = torch.sort(logits, descending=True)
                    cum_probs = torch.cumsum(F.softmax(s_logits, dim=-1), dim=-1)
                    s_mask = cum_probs > top_p
                    s_mask[..., 1:] = s_mask[..., :-1].clone()
                    s_mask[..., 0] = 0
                    mask = s_mask.scatter(1, s_indices, s_mask)
                    logits[mask] = -float("Inf")
                probs = F.softmax(logits, dim=-1)
                next_tok = torch.multinomial(probs, num_samples=1)
            else:
                next_tok = torch.argmax(logits, dim=-1, keepdim=True)

            gen = torch.cat([gen, next_tok], dim=1)
            curr_ids = next_tok
            if any(next_tok.item() == eid for eid in eos_ids):
                break

        return gen

# 8. Instantiate Elastic Model
config = BabyFlashConfig(
    vocab_size=TOTAL_VOCAB_SIZE,
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
model = BabyFlashMultiModalForCausalLM(config).to(device)

total_params = sum(p.numel() for p in model.parameters())
active_params = sum(p.numel() for n, p in model.named_parameters() if not n.startswith("model.decoder") or "shared" in n or "routed_experts.0" in n or "routed_experts.1" in n)
print(f"✓ Model '{MODEL_SCALE}' compiled successfully on device '{device.upper()}'.")
print(f"✓ Total Parameters: {total_params:,} | Active Parameters per Token: ~{active_params:,}")


In [ ]:
# @title 🚀 Cell 4: Production Training Engine with Scalable Batching & Local Release
Run_This_Cell= "4" # @param {type:"string"}
import gc
import os
import shutil
import glob
import json
from torch.utils.data import DataLoader
from safetensors.torch import save_file

# 1. Setup Optimizer and Mixed-Precision Scaler
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
use_amp = ENABLE_MIXED_PRECISION and (device == "cuda")
if device == "cuda":
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
else:
    scaler = torch.amp.GradScaler("cpu", enabled=False)

train_loader = DataLoader(tokenized_dataset, batch_size=BATCH_SIZE, shuffle=True)
steps_per_epoch = len(train_loader)
total_target_steps = EPOCHS * (steps_per_epoch // max(1, GRAD_ACCUM_STEPS))

# Cosine Learning Rate Scheduler with Warmup
warmup_steps = max(10, int(0.05 * total_target_steps))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_target_steps), eta_min=1e-6)

# 2. Inspect Existing Drive Checkpoints & Auto-Resume Logic
existing_ckpts = []
if os.path.exists(RUN_CHECKPOINT_DIR):
    for d in os.listdir(RUN_CHECKPOINT_DIR):
        if d.startswith("checkpoint-step-"):
            try:
                s_num = int(d.split("-")[-1])
                existing_ckpts.append((s_num, os.path.join(RUN_CHECKPOINT_DIR, d)))
            except ValueError:
                pass
    existing_ckpts.sort(key=lambda x: x[0])

start_step = 0
start_epoch = 0
loss_history = []

if existing_ckpts:
    latest_step, latest_ckpt_dir = existing_ckpts[-1]
    print("=" * 60)
    print(f"[✓] Existing training session found for Scale '{MODEL_SCALE}' on Dataset '{DATASET_PATH}'.")
    print(f"[✓] Resuming seamlessly from checkpoint: {latest_ckpt_dir} (Step {latest_step:,})...")
    print("=" * 60)

    try:
        weight_file = os.path.join(latest_ckpt_dir, "pytorch_model.bin")
        if os.path.exists(weight_file):
            model.load_state_dict(torch.load(weight_file, map_location=device))
        else:
            model = BabyFlashMultiModalForCausalLM.from_pretrained(latest_ckpt_dir).to(device)

        state_file = os.path.join(latest_ckpt_dir, "training_state.pt")
        if os.path.exists(state_file):
            state = torch.load(state_file, map_location=device)
            optimizer.load_state_dict(state["optimizer_state_dict"])
            if use_amp and state.get("scaler_state_dict") is not None:
                scaler.load_state_dict(state["scaler_state_dict"])
            start_step = state.get("step", latest_step)
            start_epoch = state.get("epoch", start_step // max(1, steps_per_epoch))
            loss_history = state.get("loss_history", [])

        print(f"✓ Resumed successfully! Continuing from Epoch {start_epoch + 1}, Global Step {start_step:,}...")
    except Exception as e:
        print(f"Notice: Auto-resume encountered issue ({e}). Starting fresh from step 0.")
        start_step = 0
        start_epoch = 0
else:
    print("=" * 60)
    print(f"[+] Starting fresh training run: '{UNIQUE_RUN_ID}' with {total_target_steps:,} optimization steps.")
    print("=" * 60)

# 3. Google Drive Rolling Checkpoint Saver (Resumption State Only, Limit = 3)
def save_drive_checkpoint(curr_step: int, curr_epoch: int, is_emergency: bool = False):
    tag = f"checkpoint-step-{curr_step}" if not is_emergency else f"checkpoint-emergency-step-{curr_step}"
    save_path = os.path.join(RUN_CHECKPOINT_DIR, tag)
    os.makedirs(save_path, exist_ok=True)

    # Save minimal weights for resume (safe and fast)
    torch.save(model.state_dict(), os.path.join(save_path, "pytorch_model.bin"))
    config.save_pretrained(save_path)

    # Save Training State (Optimizer, Scaler, Step, Epoch, Loss)
    state = {
        "step": curr_step,
        "epoch": curr_epoch,
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict() if use_amp else None,
        "loss_history": loss_history,
        "unique_run_id": UNIQUE_RUN_ID,
    }
    torch.save(state, os.path.join(save_path, "training_state.pt"))
    status_label = "EMERGENCY" if is_emergency else "SCHEDULED"
    print(f"\n💾 [{status_label} SYNC] Checkpoint synced to Drive: {save_path}")

    # Enforce Rolling Limit (keep top 3 to protect Drive storage)
    all_ckpts = []
    for d in os.listdir(RUN_CHECKPOINT_DIR):
        if d.startswith("checkpoint-step-"):
            try:
                s_num = int(d.split("-")[-1])
                all_ckpts.append((s_num, os.path.join(RUN_CHECKPOINT_DIR, d)))
            except ValueError:
                pass
    all_ckpts.sort(key=lambda x: x[0])

    while len(all_ckpts) > MAX_CHECKPOINTS_TO_KEEP:
        oldest_step, oldest_dir = all_ckpts.pop(0)
        shutil.rmtree(oldest_dir, ignore_errors=True)
        print(f"🧹 [DRIVE OPTIMIZER] Pruned older checkpoint: {oldest_dir}")

# 4. Final Local Package Exporter (Hugging Face Standards + Safetensors)
def export_local_model_package(export_dir: str):
    os.makedirs(export_dir, exist_ok=True)
    print("=" * 60)
    print(f"📦 Packaging Complete Hugging Face Model into: '{export_dir}'")
    print("=" * 60)

    # A. Save model.safetensors (Cloned to eliminate duplicate shared pointer errors)
    state_dict_safe = {k: v.clone().contiguous() for k, v in model.state_dict().items()}
    safetensors_path = os.path.join(export_dir, "model.safetensors")
    save_file(state_dict_safe, safetensors_path, metadata={"format": "pt"})
    print(f"  ✓ [safetensors] Saved: {safetensors_path}")

    # B. Save legacy pytorch_model.bin
    torch.save(model.state_dict(), os.path.join(export_dir, "pytorch_model.bin"))
    print("  ✓ [weights] Saved: pytorch_model.bin")

    # C. Save config.json
    config.save_pretrained(export_dir)
    print("  ✓ [config] Saved: config.json")

    # D. Save Tokenizer files
    tokenizer.save_pretrained(export_dir)
    print("  ✓ [tokenizer] Saved: tokenizer.json, tokenizer_config.json, vocab.json")

    # E. Save generation_config.json
    gen_config = {
        "bos_token_id": int(config.bos_token_id or 151643),
        "eos_token_id": int(config.eos_token_id or 151643),
        "pad_token_id": int(config.pad_token_id or 151643),
        "temperature": 0.7,
        "top_p": 0.9,
        "top_k": 50,
        "repetition_penalty": 1.25,
        "max_new_tokens": 128,
        "do_sample": True,
    }
    with open(os.path.join(export_dir, "generation_config.json"), "w", encoding="utf-8") as f:
        json.dump(gen_config, f, indent=2)
    print("  ✓ [generation] Saved: generation_config.json")

    # F. Save Standalone Architecture Source for trust_remote_code
    model_card = f"""# {MODEL_SCALE} (DeepSeek V4.1 Flash Inspired Architecture)

This repository contains an ultra-lightweight, multi-modal **Baby AI** foundation model.

## Architecture Highlights
- **Scale**: {MODEL_SCALE}
- **Domain Specialization**: {detected_domain.upper()}
- **Architecture**: Causal Encoder-Decoder with Multi-Head Latent Attention (MLA)
- **Sparse Routing**: DeepSeekMoE ({config.num_routed_experts} Routed Experts, Top-{config.num_experts_per_tok} Active + {config.num_shared_experts} Shared Expert)
- **Vocab Size**: {config.vocab_size:,}
- **Hidden Dim**: {config.hidden_size}

## Quickstart (Transformers)
```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("{export_dir}", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("{export_dir}")

prompt = "<|im_start|>user\\nHello! Who are you?<|im_end|>\\n<|im_start|>assistant\\n"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.7, repetition_penalty=1.25)
print(tokenizer.decode(outputs[0]))
```
"""
    with open(os.path.join(export_dir, "README.md"), "w", encoding="utf-8") as f:
        f.write(model_card)
    print("  ✓ [model_card] Saved: README.md")
    print(f"🎉 Complete Hugging Face release package is ready in: {export_dir}")

# 5. Training Loop with Gradient Accumulation & OOM Guardrails
model.train()
global_step = start_step
accum_step = 0

try:
    for epoch in range(start_epoch, EPOCHS):
        epoch_loss, epoch_aux = 0.0, 0.0
        successful_steps = 0
        optimizer.zero_grad()

        for batch_idx, batch in enumerate(train_loader):
            current_batch_global_step = epoch * steps_per_epoch + batch_idx
            if current_batch_global_step < start_step * GRAD_ACCUM_STEPS:
                continue

            try:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)

                labels = input_ids.clone()
                if tokenizer.pad_token_id is not None:
                    labels[labels == tokenizer.pad_token_id] = -100

                pixel_values = None
                if TASK_DOMAIN == "vision" and (batch_idx % 2 == 0):
                    pixel_values = torch.zeros((input_ids.shape[0], 3, config.image_size, config.image_size), device=device)

                autocast_device = "cuda" if device == "cuda" else "cpu"
                with torch.amp.autocast(autocast_device, enabled=use_amp):
                    outputs = model(
                        input_ids=input_ids,
                        pixel_values=pixel_values,
                        attention_mask=attention_mask,
                        labels=labels,
                    )
                    loss = outputs.loss / GRAD_ACCUM_STEPS

                scaler.scale(loss).backward()
                accum_step += 1

                epoch_loss += loss.item() * GRAD_ACCUM_STEPS
                aux_val = outputs.router_aux_loss.item() if outputs.router_aux_loss is not None else 0.0
                epoch_aux += aux_val

                # Optimizer step upon completing gradient accumulation cycle
                if accum_step % GRAD_ACCUM_STEPS == 0 or (batch_idx + 1) == len(train_loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    scheduler.step()

                    successful_steps += 1
                    global_step += 1
                    loss_history.append((global_step, round(loss.item() * GRAD_ACCUM_STEPS, 4)))

                    if global_step % 50 == 0 or global_step == 1:
                        current_lr = scheduler.get_last_lr()[0]
                        print(f"Epoch [{epoch+1}/{EPOCHS}] Step [{global_step:>5d}/{total_target_steps}] Loss: {loss.item() * GRAD_ACCUM_STEPS:.4f} (MoE Aux: {aux_val:.4f}) | LR: {current_lr:.2e}")

                    # Periodic Resume Checkpointing to Google Drive
                    if SAVE_TO_DRIVE and (global_step % SAVE_EVERY_N_STEPS == 0):
                        save_drive_checkpoint(global_step, epoch, is_emergency=False)

            except torch.cuda.OutOfMemoryError:
                print(f"⚠️ OOM intercepted at step {global_step}. Purging CUDA cache and resuming...")
                gc.collect()
                torch.cuda.empty_cache()
                optimizer.zero_grad()
                continue

        avg_loss = epoch_loss / max(1, successful_steps * GRAD_ACCUM_STEPS)
        avg_aux = epoch_aux / max(1, successful_steps * GRAD_ACCUM_STEPS)
        print(f"\n>>> Epoch {epoch+1} Complete | Average Loss = {avg_loss:.4f} | MoE Aux = {avg_aux:.4f}\n")

    # Final Checkpoint & Local Release Export
    if SAVE_TO_DRIVE:
        save_drive_checkpoint(global_step, EPOCHS, is_emergency=False)
    export_local_model_package(LOCAL_EXPORT_DIR)
    print(f"🎉 Training fully completed! Final model packaged at: {LOCAL_EXPORT_DIR}")

except KeyboardInterrupt:
    print("\n" + "!" * 60)
    print("⚠️ Training paused by user! Triggering emergency checkpoint...")
    print("!" * 60)
    if SAVE_TO_DRIVE:
        save_drive_checkpoint(global_step, epoch, is_emergency=True)
    export_local_model_package(LOCAL_EXPORT_DIR)
    print(f"✓ State safely preserved. Re-running will resume from step {global_step:,}!")


In [ ]:
# @title 💬 Cell 5: Test, Infer & Publish Playground
# @markdown Test generation across domains, verify responses, or optionally publish to Hugging Face Hub.

TEST_PROMPT = "how are you ?"  # @param {type:"string"}
MAX_NEW_TOKENS = 60  # @param {type:"integer"}
TEMPERATURE = 0.7  # @param {type:"number"}
TOP_P = 0.9  # @param {type:"number"}
TOP_K = 50  # @param {type:"integer"}
REPETITION_PENALTY = 1.25  # @param {type:"number"}

# 1. Interactive Anti-Repetition Inference Function
def test_inference(prompt: str, is_vision: bool = False):
    model.eval()
    formatted_prompt = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
    print(f"\n[Input Prompt ({detected_domain.upper()})]: {prompt}")
    input_ids = tokenizer(formatted_prompt, return_tensors="pt")["input_ids"].to(device)

    pixel_values = None
    if is_vision:
        pixel_values = torch.randn(1, 3, config.image_size, config.image_size, device=device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            pixel_values=pixel_values,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_k=TOP_K,
            top_p=TOP_P,
            repetition_penalty=REPETITION_PENALTY,
            eos_token_id=tokenizer.eos_token_id,
        )

    full_response = tokenizer.decode(output_ids[0], skip_special_tokens=False)
    assistant_reply = full_response
    if "<|im_start|>assistant\n" in full_response:
        assistant_reply = full_response.split("<|im_start|>assistant\n")[-1]
    if "<|im_end|>" in assistant_reply:
        assistant_reply = assistant_reply.split("<|im_end|>")[0]

    print(f"\n[BabyAI Output]:\n{assistant_reply.strip()}\n")
    return assistant_reply

# Run Main Interactive Test
test_inference(TEST_PROMPT, is_vision=(detected_domain == "vision"))

# Specialized Domain Verification
if detected_domain == "coding":
    print("--- Domain Code Generation Test ---")
    test_inference("Write a Python function to check if a string is a palindrome.")

elif detected_domain == "mathematics_reasoning":
    print("--- Domain Mathematical Reasoning Test (<think> tag) ---")
    test_inference("Solve for x: 3x + 12 = 36.")

elif detected_domain == "science_stem":
    print("--- Domain Science / STEM Test ---")
    test_inference("Explain Newton's third law of motion in simple terms.")

# 2. 1-Click Hugging Face Hub Publishing (Optional)
if PUSH_TO_HUB and HF_TOKEN and HF_REPO_ID:
    try:
        from huggingface_hub import HfApi, login
        print(f"\nAuthenticating with Hugging Face Hub...")
        login(token=HF_TOKEN)
        api = HfApi()
        print(f"Uploading full '{LOCAL_EXPORT_DIR}' folder to '{HF_REPO_ID}'...")
        api.upload_folder(
            folder_path=LOCAL_EXPORT_DIR,
            repo_id=HF_REPO_ID,
            repo_type="model",
            token=HF_TOKEN
        )
        print(f"🎉 Successfully published to: https://huggingface.co/{HF_REPO_ID}")
    except Exception as e:
        print(f"Notice: Hub publish encountered: {e}")
elif PUSH_TO_HUB:
    print("\nNotice: Set PUSH_TO_HUB=True with valid HF_TOKEN and HF_REPO_ID in Cell 1 to publish.")
